In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("data-modeling-gold-layer-summary")
    .getOrCreate()
)

print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 22:04:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/04 22:04:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


3.5.4


# Data Modeling Strategy Comparison

## Business Problem

The business wants a gold-layer table for sales reporting with one row per product per order.

Required fields:
- order_id
- order_date
- customer_id
- customer_name
- customer_state
- product_id
- product_name
- category
- quantity
- unit_price
- line_amount

The goal is to build the same gold-layer output from:
1. Inmon
2. Kimball
3. Data Vault 2.0

Then compare the complexity, strengths, and tradeoffs of each modeling approach.

## Inmon

In [4]:
inmon_customer = spark.read.parquet(
    str(DATA_DIR / "inmon" / "customer")
)

inmon_order = spark.read.parquet(
    str(DATA_DIR / "inmon" / "order")
)

inmon_order_item = spark.read.parquet(
    str(DATA_DIR / "inmon" / "order_item")
)

inmon_product = spark.read.parquet(
    str(DATA_DIR / "inmon" / "product")
)


oi = inmon_order_item.alias("oi")
o = inmon_order.alias("o")
c = inmon_customer.alias("c")
p = inmon_product.alias("p")


gold_from_inmon = (
    oi
    .join(o, on="order_id")
    .join(c, on="customer_id")
    .join(p, on="product_id")
    .withColumn(
        "customer_name",
        F.concat_ws(" ", F.col("c.first_name"), F.col("c.last_name"))
    )
    .withColumn(
        "line_amount",
        F.col("oi.quantity") * F.col("oi.unit_price")
    )
    .select(
        F.col("order_id"),
        F.col("o.order_date").alias("order_date"),
        F.col("customer_id"),
        F.col("customer_name"),
        F.col("c.state").alias("customer_state"),
        F.col("product_id"),
        F.col("p.product_name").alias("product_name"),
        F.col("p.category").alias("category"),
        F.col("oi.quantity").alias("quantity"),
        F.col("oi.unit_price").alias("unit_price"),
        F.col("line_amount")
    )
)

gold_from_inmon.show()

+--------+----------+-----------+-------------+--------------+----------+-------------------+-----------+--------+----------+-----------+
|order_id|order_date|customer_id|customer_name|customer_state|product_id|       product_name|   category|quantity|unit_price|line_amount|
+--------+----------+-----------+-------------+--------------+----------+-------------------+-----------+--------+----------+-----------+
|    5001|2026-03-10|       1001| Alice Nguyen|            CA|      2001|     Wireless Mouse|Electronics|       1|     34.99|      34.99|
|    5001|2026-03-10|       1001| Alice Nguyen|            CA|      2002|Mechanical Keyboard|Electronics|       1|     89.99|      89.99|
|    5002|2026-03-11|       1002|  Marcus Hill|            TX|      2003|       Laptop Stand|     Office|       2|     44.99|      89.98|
|    5003|2026-03-14|       1001| Alice Nguyen|            CA|      2004|          USB-C Hub|Electronics|       1|     54.99|      54.99|
|    5004|2026-03-18|       1004| 

#### Pros, Cons, and Best Use Cases

**Best for**
- Enterprise-wide integration
- Strongly normalized warehouse design
- Environments where data consistency and centralized governance matter
- Organizations that want a normalized EDW first, then downstream marts

**Pros**
- Strong enterprise data consistency
- Reduces redundancy through normalization
- Clear relational structure
- Good foundation for multiple downstream analytical models
- Works well when many subject areas must be integrated centrally

**Cons**
- Analytics often require many joins
- Less intuitive for BI users
- Usually needs an additional dimensional/reporting layer
- Enterprise modeling can become slow and complex
- Changes to the enterprise model can require careful coordination

**Typical use case**
A large organization builds a centralized enterprise warehouse containing normalized Customer, Order, Product, Provider, Account, etc., then publishes dimensional marts for reporting.

## Kimball

In [5]:
fact_sales = spark.read.parquet(
    str(DATA_DIR / "kimball" / "fact_sales")
)

dim_customer = spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
)

dim_product = spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_product")
)

dim_date = spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_date")
)

fs = fact_sales.alias("fs")
dc = dim_customer.alias("dc")
dp = dim_product.alias("dp")
dd = dim_date.alias("dd")

gold_from_kimball = (
    fs
    .join(dc, on="customer_key")
    .join(dp, on="product_key")
    .join(dd, on="date_key")
    .withColumn(
        "customer_name",
        F.concat_ws(
            " ",
            F.col("dc.first_name"),
            F.col("dc.last_name")
        )
    )
    .select(
        F.col("fs.order_id").alias("order_id"),
        F.col("dd.full_date").alias("order_date"),
        F.col("dc.customer_id").alias("customer_id"),
        F.col("customer_name"),
        F.col("dc.state").alias("customer_state"),
        F.col("dp.product_id").alias("product_id"),
        F.col("dp.product_name").alias("product_name"),
        F.col("dp.category").alias("category"),
        F.col("fs.quantity").alias("quantity"),
        F.col("fs.unit_price").alias("unit_price"),
        F.col("fs.line_amount").alias("line_amount")
    )
)

gold_from_kimball.show()

+--------+----------+-----------+-------------+--------------+----------+-------------------+-----------+--------+----------+-----------+
|order_id|order_date|customer_id|customer_name|customer_state|product_id|       product_name|   category|quantity|unit_price|line_amount|
+--------+----------+-----------+-------------+--------------+----------+-------------------+-----------+--------+----------+-----------+
|    5003|2026-03-14|       1001| Alice Nguyen|            CA|      2004|          USB-C Hub|Electronics|       1|     54.99|      54.99|
|    5001|2026-03-10|       1001| Alice Nguyen|            CA|      2002|Mechanical Keyboard|Electronics|       1|     89.99|      89.99|
|    5001|2026-03-10|       1001| Alice Nguyen|            CA|      2001|     Wireless Mouse|Electronics|       1|     34.99|      34.99|
|    5002|2026-03-11|       1002|  Marcus Hill|            TX|      2003|       Laptop Stand|     Office|       2|     44.99|      89.98|
|    5005|2026-03-20|       1003| 

#### Pros, Cons, and Best Use Cases

**Best for**
- BI and analytics
- Dashboarding and self-service reporting
- Business-process-oriented models
- Environments where query simplicity and usability matter most

**Pros**
- Easy for analysts and BI tools to understand
- Fewer joins for common business questions
- Facts and dimensions map naturally to reporting concepts
- Good performance for aggregation-heavy queries
- SCD patterns provide strong historical reporting capabilities

**Cons**
- Surrogate-key management must be reliable
- SCD Type 2 logic adds ETL complexity
- Denormalization intentionally duplicates descriptive data
- Multiple business processes usually require multiple fact tables
- Poor dimension design can create inconsistent or duplicated business logic

**Typical use case**
A sales warehouse contains `fact_sales`, `dim_customer`, `dim_product`, and `dim_date`, allowing Tableau or Power BI users to analyze sales with simple joins.

### Data Vault 2.0

In [6]:
hub_customer = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "hub_customer")
)

sat_customer = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_customer")
)

hub_order = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "hub_order")
)

sat_order = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_order")
)

hub_product = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "hub_product")
)

sat_product = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_product")
)

link_customer_order = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "link_customer_order")
)

link_order_product = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "link_order_product")
)

sat_order_product = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_order_product")
)

from pyspark.sql.window import Window

customer_window = (
    Window
    .partitionBy("customer_hk")
    .orderBy(F.desc("load_date"))
)

sat_customer_current = (
    sat_customer
    .withColumn(
        "rn",
        F.row_number().over(customer_window)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

hc = hub_customer.alias("hc")
sc = sat_customer_current.alias("sc")
ho = hub_order.alias("ho")
so = sat_order.alias("so")
hp = hub_product.alias("hp")
sp = sat_product.alias("sp")
lco = link_customer_order.alias("lco")
lop = link_order_product.alias("lop")
sop = sat_order_product.alias("sop")

gold_from_vault = (
    lco
    .join(hc, on="customer_hk")
    .join(sc, on="customer_hk")
    .join(ho, on="order_hk")
    .join(so, on="order_hk")
    .join(lop, on="order_hk")
    .join(sop, on="order_product_hk")
    .join(hp, on="product_hk")
    .join(sp, on="product_hk")
    .withColumn(
        "customer_name",
        F.concat_ws(
            " ",
            F.col("sc.first_name"),
            F.col("sc.last_name")
        )
    )
    .withColumn(
        "line_amount",
        F.col("sop.quantity") * F.col("sop.unit_price")
    )
    .select(
        F.col("ho.order_id").alias("order_id"),
        F.col("so.order_date").alias("order_date"),
        F.col("hc.customer_id").alias("customer_id"),
        F.col("customer_name"),
        F.col("sc.state").alias("customer_state"),
        F.col("hp.product_id").alias("product_id"),
        F.col("sp.product_name").alias("product_name"),
        F.col("sp.category").alias("category"),
        F.col("sop.quantity").alias("quantity"),
        F.col("sop.unit_price").alias("unit_price"),
        F.col("line_amount")
    )
)

gold_from_vault.show()

+--------+----------+-----------+-------------+--------------+----------+-------------------+-----------+--------+----------+-----------+
|order_id|order_date|customer_id|customer_name|customer_state|product_id|       product_name|   category|quantity|unit_price|line_amount|
+--------+----------+-----------+-------------+--------------+----------+-------------------+-----------+--------+----------+-----------+
|    5001|2026-03-10|       1001| Alice Nguyen|            CA|      2002|Mechanical Keyboard|Electronics|       1|     89.99|      89.99|
|    5001|2026-03-10|       1001| Alice Nguyen|            CA|      2001|     Wireless Mouse|Electronics|       1|     34.99|      34.99|
|    5002|2026-03-11|       1002|  Marcus Hill|            TX|      2003|       Laptop Stand|     Office|       2|     44.99|      89.98|
|    5003|2026-03-14|       1001| Alice Nguyen|            CA|      2004|          USB-C Hub|Electronics|       1|     54.99|      54.99|
|    5004|2026-03-18|       1004| 

#### Pros, Cons, and Best Use Cases

**Best for**
- Large numbers of changing source systems
- Strong historization and audit requirements
- Complex enterprise integration
- Acquisitions, source migrations, and frequent schema change
- Environments where lineage and traceability are critical

**Pros**
- Excellent preservation of historical source data
- Strong auditability and lineage
- Hubs, Links, and Satellites separate identity, relationships, and descriptive change
- New sources and attributes can often be added without redesigning the entire model
- Hash keys support deterministic and parallel loading
- Handles source-system change well

**Cons**
- Very join-heavy for analytics
- More tables and more implementation complexity
- Raw Vault is not friendly for BI users
- Historical queries require explicit handling of satellite versions
- PIT and bridge tables may be needed for performance
- Usually requires an additional business/consumption layer

**Typical use case**
A large financial, healthcare, insurance, or government organization integrates many operational systems while retaining full source history and auditability, then builds dimensional models on top for reporting.

## Mental Model

**Inmon asks:**
"What are the enterprise entities, and how should they be normalized?"

**Kimball asks:**
"What business process are we analyzing, what is the grain, and what facts and dimensions support it?"

**Data Vault asks:**
"What are the stable business identities, how are they related, and how do their attributes change over time?"

## Practical Architecture

These approaches are not necessarily mutually exclusive.

A modern platform might use:

Bronze / Raw Sources  
→ Data Vault or normalized integration layer  
→ Business transformations  
→ Kimball dimensional Gold layer  
→ Tableau / Power BI / analytics

In that architecture:

- Data Vault or Inmon handles enterprise integration
- Kimball handles analytical consumption